In [1]:
import sqlite3
import pandas as pd


# **Connect**

In [2]:
conn = sqlite3.connect("../data/nepal.sqlite")


In [3]:
# what tables are actually in here?
pd.read_sql("SELECT name FROM sqlite_schema WHERE type='table'", conn)

,name
0,building_structure
1,building_damage


# **Explore**

In [4]:
pd.read_sql("SELECT * FROM building_structure LIMIT 5", conn)

,building_id,district_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration
0,120101000011,12,1,9,288,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular
1,120101000021,12,1,15,364,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular
2,120101000031,12,1,20,384,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular
3,120101000041,12,1,20,312,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular
4,120101000051,12,1,30,308,9,Flat,Other,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular


In [5]:
pd.read_sql("SELECT * FROM building_damage LIMIT 5", conn)

,building_id,count_floors_post_eq,height_ft_post_eq,condition_post_eq,damage_grade
0,120101000011,1,9,Damaged-Used in risk,Grade 3
1,120101000021,1,9,Damaged-Repaired and used,Grade 5
2,120101000031,1,9,Damaged-Repaired and used,Grade 2
3,120101000041,1,9,Damaged-Repaired and used,Grade 2
4,120101000051,1,9,Damaged-Repaired and used,Grade 1


In [6]:
# how many buildings total, and how many districts are represented?
pd.read_sql("SELECT COUNT(*) AS n_buildings FROM building_structure", conn)

,n_buildings
0,208638


In [7]:
pd.read_sql("SELECT DISTINCT district_id FROM building_structure", conn)

,district_id
0,12
1,21
2,24
3,29


In [8]:
# which district has the most buildings? that's the one I'll focus the model on
pd.read_sql("""
    SELECT district_id, COUNT(*) AS n_buildings
    FROM building_structure
    GROUP BY district_id
    ORDER BY n_buildings DESC
""", conn)

,district_id,n_buildings
0,24,98019
1,21,58623
2,12,39352
3,29,12644


District 24 has the most observations by a comfortable margin -- going with that one
for the model, same reasoning as picking a single focused district rather than mixing
several together.

In [9]:
# quick sanity check: same building_id shows up in both tables?
query = """
    SELECT COUNT(*) AS matched_buildings
    FROM building_structure AS s
    JOIN building_damage AS d ON s.building_id = d.building_id
    WHERE s.district_id = 24
"""
pd.read_sql(query, conn)

,matched_buildings
0,98019


Matches the row count from `building_structure` for district 24, so every building
has a damage record -- no orphaned rows to worry about on the join.

In [10]:
# preview the joined data before committing to a full query
query = """
    SELECT s.*, d.damage_grade, d.count_floors_post_eq, d.height_ft_post_eq, d.condition_post_eq
    FROM building_structure AS s
    JOIN building_damage AS d ON s.building_id = d.building_id
    WHERE s.district_id = 24
    LIMIT 5
"""
pd.read_sql(query, conn)

,building_id,district_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,damage_grade,count_floors_post_eq,height_ft_post_eq,condition_post_eq
0,240101000011,24,1,40,324,12,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Grade 3,1,12,Damaged-Not used
1,240101000021,24,2,30,382,20,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Attached-1 side,Rectangular,Grade 5,0,0,Damaged-Rubble unclear
2,240101000031,24,1,13,405,10,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,Grade 5,0,0,Damaged-Rubble unclear
3,240101000041,24,2,25,328,18,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,Timber-Planck,Not attached,Rectangular,Grade 4,2,18,Damaged-Not used
4,240101000051,24,2,15,405,20,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,Grade 4,1,7,Damaged-Not used


In [11]:
# how are the damage grades distributed for this district?
query = """
    SELECT damage_grade, COUNT(*) AS n
    FROM building_damage AS d
    JOIN building_structure AS s ON s.building_id = d.building_id
    WHERE s.district_id = 24
    GROUP BY damage_grade
    ORDER BY damage_grade
"""
pd.read_sql(query, conn)

,damage_grade,n
0,Grade 1,8330
1,Grade 2,11726
2,Grade 3,25130
3,Grade 4,28974
4,Grade 5,23859


# **Import**

Wrapping the join query into a `wrangle` function so all the cleaning happens in one
place -- SQL does the heavy lifting of combining the two tables and filtering to one
district, then pandas handles the feature engineering and leakage/multicollinearity cleanup
on the result.

In [12]:
# testing the query + cleaning steps loose before wrapping into a function
query = """
    SELECT s.*, d.damage_grade, d.count_floors_post_eq, d.height_ft_post_eq, d.condition_post_eq
    FROM building_structure AS s
    JOIN building_damage AS d ON s.building_id = d.building_id
    WHERE s.district_id = 24
"""
df_test = pd.read_sql(query, conn, index_col="building_id")

df_test["damage_grade"] = df_test["damage_grade"].str[-1].astype(int)
df_test["severe_damage"] = (df_test["damage_grade"] > 3).astype(int)
df_test.shape

(98019, 17)

In [13]:
# good, wrap the whole thing into a function
def wrangle(db_path, district_id=24):
    conn = sqlite3.connect(db_path)

    # SQL does the join + district filter, pandas does the rest
    query = f"""
        SELECT s.*, d.damage_grade, d.count_floors_post_eq, d.height_ft_post_eq, d.condition_post_eq
        FROM building_structure AS s
        JOIN building_damage AS d ON s.building_id = d.building_id
        WHERE s.district_id = {district_id}
    """
    df = pd.read_sql(query, conn, index_col="building_id")
    conn.close()

    # parse the target grade out of strings like "Grade 3"
    df["damage_grade"] = df["damage_grade"].str[-1].astype(float)
    df = df.dropna(subset=["damage_grade"])
    df["damage_grade"] = df["damage_grade"].astype(int)

    # binary target: severe damage = Grade 4 or 5
    df["severe_damage"] = (df["damage_grade"] > 3).astype(int)

    # drop leaky post-earthquake columns, the old target, the district (now constant),
    # and count_floors_pre_eq (correlated 0.75 with height_ft_pre_eq -- multicollinearity)
    drop_cols = [c for c in df.columns if "post_eq" in c]
    drop_cols += ["damage_grade", "district_id", "count_floors_pre_eq"]
    df.drop(columns=drop_cols, inplace=True)

    # a couple of rows have nulls in position/plan_configuration -- just drop them
    df = df.dropna()
    return df


In [14]:
df = wrangle("../data/nepal.sqlite")
df.head()

,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,severe_damage
building_id,,,,,,,,,,,
240101000011,40,324,12,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,0
240101000021,30,382,20,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Attached-1 side,Rectangular,1
240101000031,13,405,10,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,1
240101000041,25,328,18,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,Timber-Planck,Not attached,Rectangular,1
240101000051,15,405,20,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,1


In [15]:
df.shape

(98019, 11)

In [16]:
df.isnull().sum()

age_building              0
plinth_area_sq_ft         0
height_ft_pre_eq          0
land_surface_condition    0
foundation_type           0
roof_type                 0
ground_floor_type         0
other_floor_type          0
position                  0
plan_configuration        0
severe_damage             0
dtype: int64

In [17]:
# saving the wrangled data so the EDA notebook picks up from here
df.to_csv("../data/nepal-wrangled.csv")
print("saved")

saved
